<a href="https://colab.research.google.com/github/j-see17/Digital-Analytics-Digital-Twin-/blob/main/Group_7__BUMK744_Team_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Team Assignment 2: Marketing Analytics**
*BUMK744 Marketing Research and Analytics*

Team: 7

Team members, sorted in alphabetical order by last name:
* Abraha, Heven
* Aljarrah, Yazan
* Bishop, Nia
* Hernandez, Grant
* Jarosinski, Austin
* See, Julia

By submitting this assignment, you are acknowledging the following pledge:

"I pledge on my honor that I have not given or received any unauthorized assistance on this exam/assignment."


---


## Introduction

### a) Dataset Description

The dataset was sourced from Kaggle and covers 21,613 residential property transactions between May 2014 and May 2015. It includes 21 variables spanning property characteristics, quality ratings, and location details.

#### Key Variables

- **Price (target)**
  - Range: \$75,000 – \$7.7M  
  - Median: $450,000  

  - Mean: $540,089  
  - Indicates a right-skewed distribution driven by luxury properties.

- **Size Variables**
  - `sqft_living` (290 – 13,540 sq ft)  
  - `sqft_lot`  
  - `sqft_above`  
  - `sqft_basement`

- **Property Features**
  - `bedrooms` (0 – 33)  
  - `bathrooms` (0 – 8)  
  - `floors`  
  - `year_built` (1900 – 2015)

- **Quality / Condition Ratings**
  - `grade`: overall construction quality  
  - `condition`: maintenance state  
  - Both measured on ordinal scales

- **Location Features**
  - `latitude`, `longitude`  
  - 70 unique ZIP codes  
  - `waterfront` and `view` scores capture premium location attributes  
  - Only ~0.8% of homes are waterfront

#### Data Quality

The dataset is clean with no missing values, making it well-suited for multivariate analysis without significant data cleaning.

### b) Analytical Purpose: PCA + K-Means Clustering

**Goal:** Segment the housing market into meaningful property clusters to uncover natural groupings that simple price brackets or bedroom counts would miss.

**Approach:**

- **PCA (Principal Component Analysis)** reduces the high-dimensional feature space (size, quality, age, and location variables) into a smaller set of uncorrelated principal components.  
  - This addresses multicollinearity (e.g., `sqft_living`, `sqft_above`, and `sqft_living15` are likely highly correlated).  
  - It also makes the subsequent clustering more stable and interpretable.

- **K-Means Clustering** identifies distinct market segments (e.g., entry-level suburban, mid-range family, luxury waterfront) based on the principal components.

**Managerial Value:**

- Real estate agents and investors can tailor pricing strategies and marketing by segment rather than treating the market as homogeneous.  
- Homebuyers benefit from recommendation systems that surface comparable properties within the same cluster.  
- Policymakers and developers can identify underserved segments or geographic clusters of aging or low-grade housing to guide investment or zoning decisions.

**Summary:**

This analysis moves beyond raw price prediction toward understanding the underlying structure of the housing market, making it more actionable for strategic decision-making.

## Analysis


## Real Estate Market Segmentation: PCA & K-Means Analysis

**Insight:** Traditional segmentation by price alone oversimplifies the housing market.
High correlation between sqft variables distorts direct clustering. Our goal is to
uncover natural housing market segments defined by size, location, and property features.

**Approach:** 8 features were selected (price, bedrooms, bathrooms, sqft_living, sqft_lot,
floors, view, waterfront). After removing outliers via IQR and standardizing, PCA reduces
these into 2 uncorrelated components, then K-Means identifies natural market segments.

**PCA Results:**
- **PC1 (Size/Wealth):** Driven by sqft_living, price, bedrooms, bathrooms -> captures overall property scale
- **PC2 (Location/Luxury):** Driven by waterfront, view, floors -> captures premium location value

**Optimal K = 4.** The elbow plot shows clear inflection at K=4,
balancing cluster separation with business interpretability.

**Cluster Profiles:**
| Cluster | Segment | Characteristics |
|---|---|---|
| 0 | Entry-Level | Smallest sqft, fewest bedrooms, lowest price — target for first-time buyer campaigns |
| 1 | Mid-Range | Average size and price — broadest buyer segment, ideal for general marketing |
| 2 | Upper-Mid | Larger sqft, better location scores — strong upsell opportunity |
| 3 | Luxury | Waterfront and view properties, low volume, high value — premium marketing required |

**Key Insight:** The market is structured by combinations of size, location, and property
features, not price alone.

In [ ]:
# =========================
# 1. LOAD DATA
# =========================
import pandas as pd

url = "https://raw.githubusercontent.com/nbishop1-umd/marketing-analytics-project/refs/heads/main/Housing.csv"
df = pd.read_csv(url)

# =========================
# 2. FEATURE SELECTION
# =========================
features = [
    'price',
    'bedrooms',
    'bathrooms',
    'sqft_living',
    'sqft_lot',
    'floors',
    'view',
    'waterfront'
]

df = df[features]

# =========================
# 3. DATA CLEANING
# =========================
# Convert to numeric (handles hidden issues)
df = df.apply(pd.to_numeric, errors='coerce')

# Remove missing values
df = df.dropna()

# Remove duplicates
df = df.drop_duplicates()

# Optional: Remove outliers (IQR method)
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

df = df[~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)]

# =========================
# 4. STANDARDIZATION
# =========================
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# Convert back to DataFrame (for readability)
X_scaled = pd.DataFrame(X_scaled, columns=features)

# =========================
# 5. PCA (DIMENSION REDUCTION)
# =========================
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Convert to DataFrame
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])

# Explained variance
print("Explained Variance Ratio:", pca.explained_variance_ratio_)

# =========================
# 6. ELBOW METHOD (CHOOSE K)
# =========================
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

K_range = range(1, 21)
sse = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=777, n_init=25)
    kmeans.fit(X_pca)
    sse.append(kmeans.inertia_)

plt.figure(figsize=(8,5))
plt.plot(K_range, sse, marker='o')
plt.xlabel("Number of clusters (K)")
plt.ylabel("SSE / Inertia")
plt.title("Elbow Method for Optimal K")
plt.xticks(K_range)
plt.grid(True)
plt.show()

# =========================
# 7. FINAL K-MEANS MODEL
# =========================
kmeans = KMeans(n_clusters=4, random_state=777, n_init=25)
pca_df['cluster'] = kmeans.fit_predict(X_pca)

# =========================
# 8. FINAL VISUALIZATION (CLUSTERS)
# =========================
import seaborn as sns

plt.figure(figsize=(8,5))
sns.scatterplot(
    x='PC1',
    y='PC2',
    hue='cluster',
    data=pca_df,
    palette='Set1'
)

plt.xlabel("PC1 (Size / Wealth)")
plt.ylabel("PC2 (Location / Luxury)")
plt.title("Housing Market Segments (K-Means on PCA)")
plt.grid(True)
plt.show()

## Conclusion


This analysis applied Principal Component Analysis (PCA) and K-Means clustering to identify meaningful segments within the housing market using variables such as price, bedrooms, bathrooms, square footage, floors, views, and waterfront access. After cleaning and standardizing the dataset, PCA reduced the complexity of the data into two principal components that captured the majority of variation in housing characteristics. K-Means clustering then grouped properties into four distinct market segments, revealing clear differences related to property size, wealth indicators, and luxury or location-based features.

The results demonstrate how data-driven segmentation can support more informed managerial decision making within the real estate industry. By identifying distinct housing groups, organizations can improve customer targeting, refine pricing strategies, and better align property offerings with consumer preferences. The analysis also highlights how housing characteristics such as living space, amenities, and location features influence market positioning and perceived value. These insights can help firms allocate marketing resources more efficiently, identify high-value customer segments, and support strategic investment decisions.

The clustering results suggest that differentiated marketing and pricing approaches may be more effective than generalized strategies across the entire housing market. Properties with premium characteristics, such as waterfront access or larger living areas, may require specialized positioning, while more affordable or standard properties may appeal to separate buyer groups with different priorities. Continuous use of segmentation and predictive analytics can also help organizations adapt to shifts in housing demand and changing consumer behavior over time.

Despite the value of the findings, several limitations should be acknowledged. The analysis relied on a limited set of housing variables and did not incorporate potentially important external factors such as neighborhood quality, school districts, crime rates, mortgage interest rates, or broader economic conditions. In addition, PCA reduces dimensional complexity, which can sometimes reduce interpretability of the original variables. The K-Means algorithm also requires the number of clusters to be predefined and assumes relatively uniform cluster shapes, which may not perfectly reflect real-world housing market dynamics.

Future research could strengthen the analysis by incorporating additional demographic, geographic, and economic variables to improve segmentation accuracy. Alternative clustering methods, such as hierarchical clustering or DBSCAN, could also be explored to compare segmentation outcomes and identify more complex market structures. Furthermore, integrating predictive machine learning models could provide deeper insights into housing price trends, buyer behavior, and future market demand, offering additional support for long-term strategic planning and managerial decision making.



## Team cooperation


Distribution of Labor and Task Delegation

* Abraha, Heven: Presenter, Colab conclusion and Recommendations
* Aljarrah, Yazan:
* Bishop, Nia: Presenter, Formatted slide design/ visualizations, constructed Python code, Identified Research questions for analysis
* Hernandez, Grant: Presenter, Slideshow content design/structure, Business implications analysis, Identified Housing Market Clusters
* Jarosinski, Austin: Presenter, Colab Introduction & Analysis Insights
* See, Julia: Presenter, Slide Visualizations/ Format

**Approach:**
We selected 8 relevant features and cleaned the data by removing missing values,
duplicates, and outliers via the IQR method. Data was standardized before applying
PCA to reduce dimensionality, followed by K-Means clustering.

**Challenges & Resolutions:**

- **Multicollinearity:** sqft_living and related size variables were highly correlated.
PCA resolved this by converting them into uncorrelated components before clustering.

- **Choosing K:** We tested K=1 through K=20 using the elbow method. The curve did not
have a perfectly sharp elbow, so we balanced the SSE output with business interpretability
to settle on K=4.

- **Interpreting PCA Components:** Components have no predefined label. We examined
feature loadings to name PC1 as "Size/Wealth" and PC2 as "Location/Luxury" in a way
meaningful for a real estate context.